# 02 · Whiplash index and April 1 snowpack

Full-record series per region, every winter 1950-51 onward, El Niño winters
highlighted (`ENSO_INDEX` from `src/config.py`). Descriptive only: no tests
before the freeze, and no fire data.

Inputs are built by code before this notebook runs (both cached in
`data/processed/`):

- `python -m src.whiplash`: Thornthwaite PET and SPEI-3 per ERA5-Land cell,
  log-logistic fit over 1951-2020, clipped to ±3.09, cos(latitude) regional
  means re-standardised over 1951-2020; whiplash = SPEI-3(JJA) - SPEI-3(DJF).
- `python -m src.snow`: April 1 00:00 UTC SWE from hourly ERA5-Land, glacier
  and perennial-snow cells excluded.
- `src.phase2`: the joined region x year table, written below to
  `data/processed/` and `results/tables/`.

Settings and their measured effects: `docs/methods_register.md`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import ENSO_INDEX, RESULTS
from src.describe import phase_summary
from src.figures import FIGURES, winter_series
from src.phase2 import SWE, WHIPLASH, build_table, write_tables

table = build_table(pd.read_csv(WHIPLASH), pd.read_csv(SWE))
write_tables(table)  # data/processed/ and results/tables/
print(f"{len(table)} rows, years {table['year'].min()}-{table['year'].max()}, index {ENSO_INDEX}")
table.head()

wrote C:\Users\dacek\Documents\Portfolio\hydrowhiplash-elnino26\data\processed\whiplash_swe_region_year.csv: 532 rows, 7 regions, years 1951-2026
wrote C:\Users\dacek\Documents\Portfolio\hydrowhiplash-elnino26\results\tables\whiplash_swe_region_year.csv: 532 rows, 7 regions, years 1951-2026
532 rows, years 1951-2026, index RONI


,region,year,winter,whiplash,spei3_djf,spei3_jja,swe_apr1_mm,spei3_djf_cellmean,spei3_jja_cellmean,enso_index,enso_djf,phase,strong,swe_apr1_mm_all_cells
0,PNW,1951,1950-1951,-1.561849,1.046248,-0.515601,210.869193,0.895489,-0.423842,RONI,-0.28,neutral,False,210.869193
1,PNW,1952,1951-1952,-0.522028,0.772699,0.250672,270.312784,0.661196,0.195397,RONI,0.65,el_nino,False,270.312784
2,PNW,1953,1952-1953,-1.112842,1.826232,0.713390,186.764059,1.563540,0.569329,RONI,0.40,neutral,False,186.764059
3,PNW,1954,1953-1954,0.572708,0.954383,1.527092,210.690096,0.816807,1.226896,RONI,0.63,el_nino,False,210.690096
4,PNW,1955,1954-1955,1.023461,-0.898704,0.124757,174.135257,-0.770349,0.093644,RONI,-0.12,neutral,False,174.135257


## Glacier exclusion

Mean April 1 SWE per region with glacier and perennial-snow cells excluded,
against all cells. The excluded cells add depth without year-to-year change,
so they inflate the mean while barely touching the variability.

In [2]:
glacier = table.groupby("region", sort=False)[["swe_apr1_mm", "swe_apr1_mm_all_cells"]].agg(["mean", "std"])
glacier.round(1)

swe_apr1_mm       swe_apr1_mm_all_cells      
              mean   std                  mean   std
region                                              
PNW          135.8  56.0                 135.8  56.0
BC           442.0  71.6                 592.8  73.1
AB            67.1  27.4                  69.3  27.4
AK           225.8  32.3                 711.4  32.6
YT           167.2  28.6                 474.8  27.8
NorCal        68.0  48.5                  68.0  48.5
SoCal          0.1   0.2                   0.1   0.2

## Whiplash, every year

Year Y pairs winter (Y-1)-Y with summer Y. Negative = a swing from a wetter
than usual winter to a drier than usual summer. 2026 has a winter value but
no summer yet (ERA5-Land ends July 2026).

In [3]:
from src import phase2_figures

phase2_figures.whiplash_figure(table)

WindowsPath('C:/Users/dacek/Documents/Portfolio/hydrowhiplash-elnino26/results/figures/whiplash_series_roni.png')

## April 1 SWE, every winter

SoCal is left out of this figure: its regional April 1 SWE is close to zero
in every winter. It stays in the table.

In [4]:
phase2_figures.swe_figure(table)  # regions with an April 1 normal of 10 mm or more

WindowsPath('C:/Users/dacek/Documents/Portfolio/hydrowhiplash-elnino26/results/figures/swe_april1_series_roni.png')

## Descriptive phase summary

Group means and El Niño differences, in the variable's units, as % of the
all-winter mean and in SDs of all winters. No p-values: those come after the
freeze.

In [5]:
cols = ["region", "n_el_nino", "n_strong", "n_neutral", "mean_all", "mean_el_nino", "mean_neutral",
        "nino_minus_neutral_pct", "nino_minus_neutral_sd", "strong_minus_all_pct",
        "strong_minus_all_sd", "nino_share_below_median"]
phase_summary(table[table["region"] != "SoCal"].dropna(subset=["swe_apr1_mm"]), "swe_apr1_mm")[cols].round(2)

,region,n_el_nino,n_strong,n_neutral,mean_all,mean_el_nino,mean_neutral,nino_minus_neutral_pct,nino_minus_neutral_sd,strong_minus_all_pct,strong_minus_all_sd,nino_share_below_median
0,PNW,26,7,26,135.80,118.64,137.92,-14.20,-0.34,-15.13,-0.37,0.69
1,BC,26,7,26,441.98,417.69,455.75,-8.61,-0.53,-6.58,-0.41,0.65
2,AB,26,7,26,67.10,53.06,72.29,-28.66,-0.70,-22.03,-0.54,0.69
3,AK,26,7,26,225.78,222.74,227.09,-1.93,-0.13,-7.50,-0.52,0.58
4,YT,26,7,26,167.22,161.21,171.12,-5.92,-0.35,-5.12,-0.30,0.62
5,NorCal,26,7,26,67.96,83.40,52.14,45.99,0.64,34.92,0.49,0.35


In [6]:
wcols = ["region", "n_el_nino", "n_strong", "mean_el_nino", "mean_neutral", "mean_strong",
         "nino_minus_neutral", "nino_minus_neutral_sd", "strong_minus_all_sd"]
w = table.dropna(subset=["whiplash"])
pd.concat({v: phase_summary(w, v)[wcols].set_index("region")
           for v in ["spei3_djf", "spei3_jja", "whiplash"]}, axis=1).round(2)

spei3_djf                                                 \
       n_el_nino n_strong mean_el_nino mean_neutral mean_strong   
region                                                            
PNW           26        7        -0.28         0.12        0.04   
BC            26        7        -0.28         0.27        0.23   
AB            26        7        -0.39         0.20       -0.25   
AK            26        7        -0.22        -0.19       -1.05   
YT            26        7        -0.24         0.06       -0.28   
NorCal        26        7         0.24        -0.18        0.60   
SoCal         26        7         0.51        -0.37        0.75   

                                                                    spei3_jja  \
       nino_minus_neutral nino_minus_neutral_sd strong_minus_all_sd n_el_nino   
region                                                                          
PNW                 -0.40                 -0.39                0.10        26   
BC                  -0.54                 -0.49                0.29        26   
AB                  -0.59                 -0.53               -0.19        26   
AK                  -0.03                 -0.02               -0.98        26   
YT                  -0.30                 -0.29               -0.27        26   
NorCal               0.42                  0.43                0.66        26   
SoCal                0.88                  0.88                0.80        26   

                 ...                                            whiplash  \
       n_strong  ... nino_minus_neutral_sd strong_minus_all_sd n_el_nino   
region           ...                                                       
PNW           7  ...                  0.16                0.25        26   
BC            7  ...                  0.31               -0.27        26   
AB            7  ...                  0.40                0.14        26   
AK            7  ...                 -0.16                0.43        26   
YT            7  ...                 -0.10               -0.32        26   
NorCal        7  ...                  0.20                0.42        26   
SoCal         7  ...                 -0.07                0.17        26   

                                                                          \
       n_strong mean_el_nino mean_neutral mean_strong nino_minus_neutral   
region                                                                     
PNW           7         0.44        -0.13        0.11               0.58   
BC            7         0.43        -0.45       -0.62               0.88   
AB            7         0.60        -0.43        0.30               1.03   
AK            7         0.15         0.29        1.47              -0.14   
YT            7         0.13        -0.07       -0.18               0.19   
NorCal        7        -0.08         0.12       -0.27              -0.20   
SoCal         7        -0.50         0.45       -0.67              -0.95   

                                                  
       nino_minus_neutral_sd strong_minus_all_sd  
region                                            
PNW                     0.42                0.12  
BC                      0.65               -0.44  
AB                      0.69                0.25  
AK                     -0.09                0.93  
YT                      0.13               -0.04  
NorCal                 -0.16               -0.16  
SoCal                  -0.70               -0.46  

[7 rows x 24 columns]